# GPT Paper Extractor

This notebook extracts TLC retention-factor (Rf) data from a paper's supporting-information PDF. Reusable functions are maintained in `GPT_paper_extractor.py`.

## 1. Import the extraction workflow

Import the reusable extraction, review, post-processing, OPSIN, and saving functions from the companion Python module.

In [ ]:
import os
from getpass import getpass

from IPython.display import display
from openai import OpenAI

from GPT_paper_extractor import (
    DEFAULT_OPSIN_JAR,
    DEFAULT_OUTPUT_DIR,
    DEFAULT_SI_DIR,
    clean_extracted_data,
    convert_iupac_to_smiles,
    deduplicate_extracted_data,
    extract_data_GPT,
    get_last_saved_metadata,
    prepare_export_data,
    remove_rows,
    save_next_datatable,
    suggested_deletion_indices,
)

## 2. Configure the OpenAI client

The API key is read securely from the `OPENAI_API_KEY` environment variable. If it is not already set, the notebook asks for it without displaying the entered value.

In [ ]:
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()

## 3. Describe the paper and input file

Set the supporting-information PDF filename and publication metadata. Place the PDF in the `SI` directory unless you change `si_dir`.

In [ ]:
pdf_file = "10.pdf"
title = "Catalytic Friedel–Crafts Alkylative Desymmetrization of Cyclohexa-2,5-dienones: Access to Linear and Bridged Polycyclic Pyrroles and 3-Arylpyrroles"
doi = "https://pubs.acs.org/doi/10.1021/acs.orglett.2c02135"
publish = "2022.24.29"

si_dir = DEFAULT_SI_DIR
opsin_jar = DEFAULT_OPSIN_JAR
output_dir = DEFAULT_OUTPUT_DIR

## 4. Extract and review Rf data

Detect pages related to Rf values, send their text to GPT one page at a time, and display the extracted table. IUPAC-name matches are highlighted in orange, while source lines containing Rf information are highlighted in green.

In [ ]:
extraction_result = extract_data_GPT(
    pdf_file,
    client=client,
    si_dir=si_dir,
    review=True,
)

if extraction_result is None:
    raise RuntimeError("No valid Rf data were extracted from the PDF.")

data_extracted, false_page_idx = extraction_result
print(f"Extracted rows: {len(data_extracted)}")
display(data_extracted)

## 5. Remove incorrect rows after manual review

The suggested indices are names that were not matched exactly in the PDF text. Review the colored output above, then enter only the indices that should actually be removed.

In [ ]:
suggested_rows = suggested_deletion_indices(false_page_idx)
print("Rows suggested for review:", suggested_rows)

# Replace this empty list with confirmed incorrect row indices.
rows_to_delete = []
data_reviewed = remove_rows(data_extracted, rows_to_delete)

print(f"Rows after manual review: {len(data_reviewed)}")
display(data_reviewed)

## 6. Clean and deduplicate the extracted records

Remove rows missing required TLC fields, construct the original composite `data_ID`, and keep one copy of each duplicated record.

In [ ]:
data_clean = clean_extracted_data(data_reviewed)
data_unique = deduplicate_extracted_data(data_clean)

print(f"Rows after cleaning and deduplication: {len(data_unique)}")
display(data_unique)

## 7. Convert IUPAC names to SMILES with OPSIN

Run the local OPSIN command-line JAR for each IUPAC name. Records that OPSIN cannot convert are excluded from the resulting table.

In [ ]:
data_with_smiles = convert_iupac_to_smiles(
    data_unique,
    jar_path=opsin_jar,
)

print(f"Rows successfully converted by OPSIN: {len(data_with_smiles)}")
display(data_with_smiles)

## 8. Attach publication metadata

Add the paper title, DOI, and publication identifier to every extracted record before export.

In [ ]:
data_for_export = prepare_export_data(
    data_with_smiles,
    title=title,
    doi=doi,
    publish=publish,
)

display(data_for_export)

## 9. Check the previous export

Display the latest saved paper metadata before writing a new table. The save function also blocks a DOI that matches the most recent export.

In [ ]:
last_title, last_doi = get_last_saved_metadata(output_dir)
print("Last saved title:", last_title)
print("Last saved DOI:", last_doi)
print("Current title:", title)
print("Current DOI:", doi)

## 10. Save the final CSV

Change `SAVE_RESULTS` to `True` only after reviewing the complete table and metadata. The output is assigned the next available `datatableN.csv` filename.

In [ ]:
SAVE_RESULTS = False

if SAVE_RESULTS:
    saved_path = save_next_datatable(
        data_for_export,
        output_dir=output_dir,
        prevent_duplicate_doi=True,
    )
    print(f"Saved: {saved_path}")
else:
    print("Save skipped. Set SAVE_RESULTS = True after final review.")